In [0]:
--  set catalog + schema
USE CATALOG ymotorna_DB;
USE SCHEMA superstore_lakehouse;

-- check
SELECT current_catalog(), 
    current_schema();   

current_catalog(),current_schema()
ymotorna_db,superstore_lakehouse


---
### Create bronze layer

In [0]:
-- drop table if exists bronze_orders;

-- + Delta tbl for bronze layer \\  changes col names + ingestion_timestamp + source_file_name
create table if not exists bronze_orders as(
    select 
         `Row ID` as row_id,
        `Order ID` as order_id,
        try_cast(`Order Date` as date) as order_date,
        try_cast(`Ship Date` as date) as ship_date,
        `Ship Mode` as ship_mode,
        `Customer ID` as customer_id,
        `Customer Name` as customer_name,
        `Segment` as segment,
        `Country` as country,
        `City` as city,
        `State` as state,
        try_cast(`Postal Code` as int) as postal_code,
        `Region` as region,
        `Product ID` as product_id,
        `Category` as category,
        `Sub-Category` as sub_category,
        `Product Name` as product_name,
        try_cast(`Sales` as double) as sales,
        try_cast(`Quantity` as int) as quantity,
        try_cast(`Discount` as double) as discount,
        try_cast(`Profit` as double) as profit,
        current_timestamp() as ingestion_timestamp,
        input_file_name() as source_file_name
    from  read_files(
    '/Volumes/ymotorna_db/superstore_lakehouse/raw_data/initial/',
    format => 'csv',
    header => true,
    inferSchema => true
    )
);


--  check
show tables;


database,tableName,isTemporary
superstore_lakehouse,bronze_orders,false
superstore_lakehouse,gold_customer_metrics,false
superstore_lakehouse,gold_sales_category,false
superstore_lakehouse,gold_sales_daily,false
superstore_lakehouse,gold_sales_region,false
superstore_lakehouse,products,false
superstore_lakehouse,silver_orders,false
superstore_lakehouse,silver_rejected_orders,false



Check

In [0]:
select count(*) from bronze_orders;
-- should be 9994


count(*)
10000


In [0]:
select * from bronze_orders limit 5;

row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,ingestion_timestamp,source_file_name
9998,CA-2024-100001,null,2024-01-08,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.91,2026-06-29T15:51:03.216Z,dbfs:/Volumes/ymotorna_db/superstore_lakehouse/raw_data/incremental/day_1.csv
9999,CA-2024-100001,null,2024-01-08,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,Hon Deluxe Fabric Upholstered Stacking Chairs,731.94,3,0.0,219.58,2026-06-29T15:51:03.216Z,dbfs:/Volumes/ymotorna_db/superstore_lakehouse/raw_data/incremental/day_1.csv
10000,CA-2024-100002,null,2024-01-10,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters,14.62,2,0.0,6.87,2026-06-29T15:51:03.216Z,dbfs:/Volumes/ymotorna_db/superstore_lakehouse/raw_data/incremental/day_1.csv
10001,CA-2024-100003,null,2024-01-11,First Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5,0.45,-383.03,2026-06-29T15:51:03.216Z,dbfs:/Volumes/ymotorna_db/superstore_lakehouse/raw_data/incremental/day_1.csv
10002,CA-2024-100004,null,2024-01-12,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold N Roll Cart System,22.37,2,0.0,2.52,2026-06-29T15:51:03.216Z,dbfs:/Volumes/ymotorna_db/superstore_lakehouse/raw_data/incremental/day_1.csv


---
### Add incremental loading

In [0]:
-- new data uploaded in separate folder 'incremental' -> change path -> dont double-upload initial data used to +tbl

copy into bronze_orders
from (
    select
        `Row ID` as row_id,
        `Order ID` as order_id,
        try_cast(`Order Date` as date) as order_date,
        try_cast(`Ship Date` as date) as ship_date,
        `Ship Mode` as ship_mode,
        `Customer ID` as customer_id,
        `Customer Name` as customer_name,
        `Segment` as segment,
        `Country` as country,
        `City` as city,
        `State` as state,
        try_cast(`Postal Code` as int) as postal_code,
        `Region` as region,
        `Product ID` as product_id,
        `Category` as category,
        `Sub-Category` as sub_category,
        `Product Name` as product_name,
        try_cast(`Sales` as double) as sales,
        try_cast(`Quantity` as int) as quantity,
        try_cast(`Discount` as double) as discount,
        try_cast(`Profit` as double) as profit,
        current_timestamp() as ingestion_timestamp,
        _metadata.file_path as source_file_name
    from '/Volumes/ymotorna_db/superstore_lakehouse/raw_data/incremental/'
)
fileformat = csv
format_options (
    'header' = 'true',
    'inferSchema' = 'true'
);

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
5,5,0


Check

In [0]:
select count(*) from bronze_orders  -- should >#

count(*)
10005


In [0]:
describe history bronze_orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-06-29T15:52:12.000Z,73808821626549,ymotorna@kse.org.ua,COPY INTO,Map(statsOnLoad -> true),null,List(1817384817085704),6dba8b69-bd38-4b20-adae-5906d84fe00a,0629-101017-f0mg8rm1-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 7271, numSkippedCorruptFiles -> 0)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
1,2026-06-29T15:51:05.000Z,73808821626549,ymotorna@kse.org.ua,COPY INTO,Map(statsOnLoad -> true),null,List(1817384817085704),f6a13c34-ee20-4170-a432-4f10f9b6d8d0,0629-101017-f0mg8rm1-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 6, numOutputBytes -> 7420, numSkippedCorruptFiles -> 0)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
0,2026-06-29T15:43:31.000Z,73808821626549,ymotorna@kse.org.ua,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1817384817085704),1e541a39-9752-4816-a24a-492fe678ae0a,0629-101017-f0mg8rm1-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 9994, numOutputBytes -> 311517)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
